# 一.导入所需要的包

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from glob import glob
import os

# 二.读取所有数据集

In [2]:
dfs={}
data_path = r'C:\Users\wangg\PycharmProjects\olist订单物流延误预测\数据集\raw_data'
dfs['customer'] = pd.read_csv(f'{data_path}/olist_customers_dataset.csv')
dfs['order'] = pd.read_csv(f'{data_path}/olist_orders_dataset.csv')
dfs['product'] = pd.read_csv(f'{data_path}/olist_products_dataset.csv')
dfs['geolocation'] = pd.read_csv(f'{data_path}/olist_geolocation_dataset.csv')
dfs['order_items'] = pd.read_csv(f'{data_path}/olist_order_items_dataset.csv')
dfs['order_payments'] = pd.read_csv(f'{data_path}/olist_order_payments_dataset.csv')
dfs['order_reviews'] = pd.read_csv(f'{data_path}/olist_order_reviews_dataset.csv')
dfs['sellers'] = pd.read_csv(f'{data_path}/olist_sellers_dataset.csv')
dfs['product_category_translation'] = pd.read_csv(f'{data_path}/product_category_name_translation.csv')

# 三.数据基本信息与描述性统计

In [3]:
for name,df in dfs.items():
    print("="*20+name+"="*20)
    print(df.info())
    print(df.describe())

====================customer====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None
       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%                24416.000000
75%                58900.000000
max                99990.000000
====================order====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440


观察结果:
1.订单数量为99441条
2.产品数量为32951个
3.客户数量为99441个
4.商家数量为30951个
5.订单评价数量为99224个,可以支撑后续相关分析，平均评分为4.086421

# 四.数据缺失值检测

In [4]:
for name,df in dfs.items():
    print("="*20+name+"="*20)
    print(df.isnull().sum())
    missing=df.isnull().sum()/len(df)
    missing=missing[missing>0]
    if len(missing)==0:
        print("无缺失值")
    print("="*20+"缺失率"+"="*20)
    print(missing)

====================customer====================
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
无缺失值
====================缺失率====================
Series([], dtype: float64)
====================order====================
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
====================缺失率====================
order_approved_at                0.001609
order_delivered_carrier_date     0.017930
order_delivered_customer_date    0.029817
dtype: float64
====================product====================
product_id                      0
product_category_name         610
product_name_lenght           610
product_descri

观察结果：
1.顾客表中无缺失值
2.订单表中，订单达成时间和商品发货时间以及商品实际送达时间存在缺失值，缺失率均低于5%,因为这些列都是流程类时间，所以可以使用相关联时间填充，若有计划送达时间，实际送达时间则使用计划送达时间填充，否则使用关联时间加减对应的平均跨度，订单达成时间可以使用购买时间填充，订单发货时间则使用购买时间加平均购买到发货时间跨度填充
3.产品表中，产品类别名字，名字长度，描述长度，照片数量存在缺失值，缺失率低于5%，缺失值可以用众数填充或者忽略，照片数量缺失值建议填充为0,体重与长度各列缺失两个，缺失值建议用众数填充或者忽略
10.订单支付表中订单支付时间列存在缺失值，缺失率低于5%，缺失值建议用众数填充或者忽略
4.商家表中无缺失值
5.订单细则表中不存在缺失值
6.订单支付表中无缺失值
7.订单评价表中订单评价标题与评价内容列存在缺失值，对后续分析预测无影响，忽略
8.地理位置表中无缺失值
9.产品类别翻译表中无缺失值

# 五.时间列格式转换

In [5]:
order_time_col=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in order_time_col:
    dfs['order'][col]=pd.to_datetime(dfs['order'][col],errors='coerce')
dfs['order_items']['shipping_limit_date']=pd.to_datetime(dfs['order_items']['shipping_limit_date'],errors='coerce')
print(dfs['order'].info())
print(dfs['order_items'].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype

# 六.创造所需字段以及合并表

## 1.合并所需要的表

In [6]:
# 合并分析总表，添加字段
df_all=pd.merge(dfs['order'],dfs['order_items'],on='order_id')
df_all=pd.merge(df_all,dfs['product'],on='product_id')
df_all=pd.merge(df_all,dfs['customer'],on='customer_id')
df_all=pd.merge(df_all,dfs['sellers'],on='seller_id')
df_all=pd.merge(df_all,dfs['product_category_translation'],on='product_category_name')

## 2.创建所需的新字段

In [7]:
df_all['volume']=df_all['product_weight_g']*df_all['product_length_cm']*df_all['product_height_cm']*df_all['product_width_cm']
df_all['weight']=df_all['product_weight_g']/1000
df_all['process_time']=(df_all['order_approved_at']-df_all['order_purchase_timestamp']).dt.total_seconds()/3600
df_all['purchase_hour']=df_all['order_purchase_timestamp'].dt.hour
df_all['purchase_month']=df_all['order_purchase_timestamp'].dt.month
df_all['purchase_dayofweek']=df_all['order_purchase_timestamp'].dt.dayofweek
# 按照订单id加总每个订单的费用以及运费
df_all['price_total']=df_all.groupby('order_id')['price'].transform('sum')
df_all['freight_total']=df_all.groupby('order_id')['freight_value'].transform('sum')


## 3.计算订单运输距离

In [8]:
dfs['geolocation']=dfs['geolocation'].groupby('geolocation_zip_code_prefix').agg({'geolocation_lat':'mean','geolocation_lng':'mean'}).reset_index()
geo = dfs['geolocation'][['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']]
# 创建客户经纬度
df_all=pd.merge(df_all,geo,left_on='customer_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')
df_all.rename(columns={'geolocation_lat':'customer_lat','geolocation_lng':'customer_lng'},inplace=True)
df_all.drop(columns=['geolocation_zip_code_prefix'], inplace=True)
# 创建商家经纬度
df_all=pd.merge(df_all,geo,left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')
df_all.rename(columns={'geolocation_lat':'seller_lat','geolocation_lng':'seller_lng'},inplace=True)
def distance(lat1,lon1,lat2,lon2):
    lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2])
    dlon=lon2-lon1
    dlat=lat2-lat1
    a=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c=2*np.arcsin(np.sqrt(a))
    return c*6371
df_all['distance']=df_all.apply(lambda x:distance(x['seller_lat'],x['seller_lng'],x['customer_lat'],x['customer_lng']),axis=1)

## 4.提取产品类别列以及州列的高层特征

In [9]:
# 州代码到地理区域的映射字典
state_to_region = {
    # 北部地区 (North) - 7个州
    'AC': 'North',  # Acre
    'AM': 'North',  # Amazonas
    'AP': 'North',  # Amapá
    'PA': 'North',  # Pará
    'RO': 'North',  # Rondônia
    'RR': 'North',  # Roraima
    'TO': 'North',  # Tocantins
    # 东北地区 (Northeast) - 9个州
    'AL': 'Northeast',  # Alagoas
    'BA': 'Northeast',  # Bahia
    'CE': 'Northeast',  # Ceará
    'MA': 'Northeast',  # Maranhão
    'PB': 'Northeast',  # Paraíba
    'PE': 'Northeast',  # Pernambuco
    'PI': 'Northeast',  # Piauí
    'RN': 'Northeast',  # Rio Grande do Norte
    'SE': 'Northeast',  # Sergipe
    # 中西部地区 (Central-West) - 4个州
    'DF': 'CentralWest',  # Distrito Federal
    'GO': 'CentralWest',  # Goiás
    'MT': 'CentralWest',  # Mato Grosso
    'MS': 'CentralWest',  # Mato Grosso do Su
    # 东南地区 (Southeast) - 4个州
    'ES': 'Southeast',    # Espírito Santo
    'MG': 'Southeast',    # Minas Gerais
    'RJ': 'Southeast',    # Rio de Janeiro
    'SP': 'Southeast',    # São Paulo
    # 南部地区 (South) - 3个州
    'PR': 'South',        # Paraná
    'RS': 'South',        # Rio Grande do Sul
    'SC': 'South',        # Santa Catarina
}
df_all['seller_region']=df_all['seller_state'].map(state_to_region)
df_all['customer_region']=df_all['customer_state'].map(state_to_region)
# df_all['is_same_region']=(df_all['seller_region']==df_all['customer_region']).astype(int)
#商品类别映射
product_category_mapping = {
    # ========== 1. 电子产品 (Electronics) ==========
    'computers_accessories': 'Electronics',
    'computers': 'Electronics',
    'electronics': 'Electronics',
    'telephony': 'Electronics',
    'fixed_telephony': 'Electronics',
    'tablets_printing_image': 'Electronics',
    'audio': 'Electronics',
    'cds_dvds_musicals': 'Electronics',
    'dvds_blu_ray': 'Electronics',
    'consoles_games': 'Electronics',
    'cine_photo': 'Electronics',
    'music': 'Electronics',
    'musical_instruments': 'Electronics',  # 合并乐器到电子产品

    # ========== 2. 家居用品 (Home & Living) ==========
    'furniture_decor': 'HomeLiving',
    'furniture_bedroom': 'HomeLiving',
    'furniture_living_room': 'HomeLiving',
    'furniture_mattress_and_upholstery': 'HomeLiving',
    'home_confort': 'HomeLiving',
    'home_comfort_2': 'HomeLiving',
    'home_construction': 'HomeLiving',
    'bed_bath_table': 'HomeLiving',
    'kitchen_dining_laundry_garden_furniture': 'HomeLiving',
    'la_cuisine': 'HomeLiving',
    'housewares': 'HomeLiving',
    'home_appliances': 'HomeLiving',
    'home_appliances_2': 'HomeLiving',
    'small_appliances': 'HomeLiving',
    'small_appliances_home_oven_and_coffee': 'HomeLiving',
    'air_conditioning': 'HomeLiving',
    'flowers': 'HomeLiving',  # 鲜花归入家居

    # ========== 3. 工具建材 (Tools & Construction) ==========
    'construction_tools_construction': 'ToolsConstruction',
    'construction_tools_lights': 'ToolsConstruction',
    'construction_tools_safety': 'ToolsConstruction',
    'costruction_tools_garden': 'ToolsConstruction',
    'costruction_tools_tools': 'ToolsConstruction',
    'garden_tools': 'ToolsConstruction',
    'signaling_and_security': 'ToolsConstruction',

    # ========== 4. 时尚服饰 (Fashion) ==========
    'fashion_shoes': 'Fashion',
    'fashion_male_clothing': 'Fashion',
    'fashio_female_clothing': 'Fashion',
    'fashion_childrens_clothes': 'Fashion',
    'fashion_underwear_beach': 'Fashion',
    'fashion_bags_accessories': 'Fashion',
    'fashion_sport': 'Fashion',
    'watches_gifts': 'Fashion',
    'luggage_accessories': 'Fashion',

    # ========== 5. 美妆健康 (Beauty & Health) ==========
    'perfumery': 'BeautyHealth',
    'health_beauty': 'BeautyHealth',
    'diapers_and_hygiene': 'BeautyHealth',
    'baby': 'BeautyHealth',  # 婴儿用品归入美妆健康（母婴类）

    # ========== 6. 书籍文具 (Books & Stationery) ==========
    'books_general_interest': 'BooksStationery',
    'books_technical': 'BooksStationery',
    'books_imported': 'BooksStationery',
    'stationery': 'BooksStationery',
    'art': 'BooksStationery',
    'arts_and_craftmanship': 'BooksStationery',

    # ========== 7. 宠物用品 (Pet Supplies) ==========
    'pet_shop': 'PetSupplies',

    # ========== 8. 玩具礼品 (Toys & Gifts) ==========
    'toys': 'ToysGifts',
    'cool_stuff': 'ToysGifts',
    'christmas_supplies': 'ToysGifts',
    'party_supplies': 'ToysGifts',

    # ========== 9. 体育户外 (Sports & Leisure) ==========
    'sports_leisure': 'SportsLeisure',

    # ========== 10. 其他 (Other) ==========
    'auto': 'Other',
    'food': 'Other',
    'food_drink': 'Other',
    'drinks': 'Other',
    'agro_industry_and_commerce': 'Other',
    'industry_commerce_and_business': 'Other',
    'market_place': 'Other',
    'office_furniture': 'Other',
    'security_and_services': 'Other',
}
df_all['product_category_high']=df_all['product_category_name_english'].map(product_category_mapping)

使用商家和客户是否在同一地区作为特征对模型没有提升，其信息已经被距离特征以及商家和卖家的地区特征展现，所以删除

## 5.创建目标字段：订单是否准时送达

In [10]:
# 查看除delivered状态的订单，是否最终送达时间为缺失
print(df_all.groupby('order_status')['order_delivered_customer_date'].agg(['count','size',lambda x:x.isnull().sum()]))

               count    size  <lambda_0>
order_status                            
approved           0       3           3
canceled           7     526         519
delivered     108630  108638           8
invoiced           0     347         347
processing         0     344         344
shipped            0    1158        1158
unavailable        0       7           7


除已送达状态的订单，其他状态订单最终送达时间几乎全部缺失，所以讲其他状态订单以及八个已送达状态但最终送达时间缺失的列全部删去

In [11]:
df_all=df_all.dropna(subset=['order_delivered_customer_date','order_estimated_delivery_date'])
print(df_all[['order_delivered_customer_date','order_estimated_delivery_date']].isnull().sum())

order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


In [12]:
df_all['is_delay']=(df_all['order_delivered_customer_date']>df_all['order_estimated_delivery_date']).astype(int)

## 6.把所有需要的字段提取出来

In [13]:
df=df_all[['order_id','customer_id','customer_unique_id','customer_region',
          'seller_id','seller_region','product_id','product_category_high','volume','weight','process_time','is_delay','distance','freight_total','price_total','purchase_hour','purchase_dayofweek','purchase_month','is_same_region'
]]
df=df.drop_duplicates()
# 数据导出
df.to_csv(r'C:\Users\wangg\PycharmProjects\olist订单物流延误预测\数据集\data_processed/data_clean.csv',index=False)